# **Agent's tools**

## **Imports and environment configs**

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI

MODEL = "gpt-5.4-mini"

openai_client = OpenAI()

In [3]:
import json

from bibliometric_rag_assistant.ingestion.pubmed import search_pubmed
from bibliometric_rag_assistant.search_strategy import PUBMED_QUERY

## **Simple Agent - answers without tools**

In [10]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history
)

print(response.output_text)

“Agent” has a few meanings, but **properly speaking** in the strongest general sense, an agent is:

> **An entity that acts on behalf of some goal, policy, or principal by selecting actions in an environment over time.**

A more precise version:

> **An agent is something that perceives conditions, chooses actions, and thereby causes changes in the world in pursuit of some objective.**

### Core ideas usually included
An agent typically has:

- **A boundary**: it is distinct from its environment
- **Perception / information intake**: it gets inputs about the world
- **Action / intervention**: it can affect the world
- **A decision rule**: it chooses what to do, not merely reacts mechanically
- **Some notion of purpose**: explicit or implicit goals, utility, preferences, or a principal’s instructions

### In different fields
- **Philosophy / law**: an agent is a being capable of intentional action and sometimes moral responsibility.
- **Economics**: an agent is a decision-maker with pre

In [11]:
history += response.output

history.append({"role": "user", "content": "What specifically is an AI agent?"})

second_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(second_response.output_text)

An **AI agent** is a system that:

1. **perceives** information from an environment,
2. **decides** what to do based on that information and its objective,
3. **takes actions** that affect the environment,
4. and often does this **autonomously over multiple steps**.

### Simple definition
> **An AI agent is software or a machine that can choose actions toward a goal, rather than only producing a one-off output.**

### What makes it an “agent” rather than just a model?
A plain model might answer a prompt once.

An AI agent usually has some combination of:

- **memory** of prior steps
- **planning**
- **tool use** (search, code execution, APIs, databases)
- **feedback loops** where it observes results and adjusts
- **goal-directed behavior**
- **some autonomy** in deciding next actions

### Example
- **Not really an agent**: a chatbot that only replies to each message independently.
- **More like an agent**: a system that can:
  - read your request,
  - break it into tasks,
  - search th

In [12]:
history += second_response.output

history.append({"role": "user", "content": "According with the last answers, will you become an agent when i provide tools and call you in a loop?"})

third_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(third_response.output_text)

Yes — **in that setup, I would function as an agent-like system**.

More precisely:

- **The base model alone** is not usually considered an agent.
- **If you wrap me in a loop** where I can:
  - observe state,
  - choose an action,
  - use tools,
  - see the results,
  - and repeat until a goal is reached,

then the overall system is an **AI agent**.

### Important nuance
It’s usually not *the model by itself* that is the agent; it’s the **model + tool access + control loop + objective**.

So if you:
1. give me tools,
2. let me decide which tool to use,
3. feed tool results back to me,
4. and keep iterating,

then I become the **policy/decision component** inside an agentic loop.

### But there’s a caveat
I still wouldn’t become a fully autonomous agent in the human sense unless the system also gave me things like:

- persistent memory,
- ability to initiate actions on my own,
- stable goals,
- long-horizon planning,
- and authority to act without step-by-step human prompting.

Withou

## **Defining a tool**

In [24]:
search_pubmed_tool = {
    "type": "function",
    "name": "search_pubmed",
    "description": (
        "Search PubMed and return the total number of matching records "
        "and a limited list of PMIDs. Use this for PubMed literature searches."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "PubMed query using its Boolean operators and field tags, "
                    "such as [Title/Abstract] and [MeSH Terms]."
                ),
            },
            "limit": {
                "type": "integer",
                "minimum": 1,
                "maximum": 20,
                "description": (
                    "Maximum number of PMIDs to return. "
                    "Use 20 when the user does not specify a limit."
                ),
            },
        },
        "required": ["query", "limit"],
        "additionalProperties": False,
    },
    "strict": True,
}


## **Sending the question with the tool**

Mesmo com a ferramenta, vai responder "na lata". Não precisa de decidir se chama o não a ferramenta, já que a pergunta não tem a menor relação com o que a ferramenta pode fazer. 

In [25]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

message ResponseOutputMessage(id='msg_0f3f399a940096fa006ac82e7ab6b887d2aacc6a3217c83170', content=[ResponseOutputText(annotations=[], text='“Agent” can mean a few different things, but **properly** in the most general sense, an agent is:\n\n> **Something that acts on behalf of or in response to something else, with some degree of autonomy and goal-directed behavior.**\n\nA few useful versions of the idea:\n\n### In philosophy / everyday language\nAn agent is a **being or system that can initiate actions** rather than merely be passively moved.  \n- A person choosing to speak is an agent.\n- A thermostat is usually not called an agent in the strongest sense, because it has very limited autonomy.\n- A dog, a company, or a government can be called agents in some contexts if they act intentionally or represent someone.\n\n### In AI / computer science\nAn agent is typically a system that:\n1. **Perceives** its environment,\n2. **Decides** what to do,\n3. **Acts** to influence the environme

Agora a pergunta terá relação com a funcionalidade da ferramenta, então na primeira chamada o modelo aprende o `args` que deve ser usado numa eventual chamada da ferramenta, mas a chamada em si só ocorre na segunda interação com a api.

- **Primeira chamada: o modelo decide se usa ou não uma tool**

In [26]:
instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


history = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": "Which pmids returns from the query in effect for this pipeline?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

function_call ResponseFunctionToolCall(arguments='{"query":"( \\"Temporomandibular Joint\\"[Mesh] OR \\"temporomandibular joint\\"[Title/Abstract] ) AND ( \\"Joint Prosthesis\\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \\"2000/01/01\\"[Date - Publication] : \\"2025/12/31\\"[Date - Publication] )","limit":20}', call_id='call_gm8g9gCOpBXK9Q5z6m6m1aeq', name='search_pubmed', type='function_call', id='fc_0ba70aae073d5cac006ac82e802b8c87d2af69e377261dbeb7', async_=None, caller=None, namespace=None, status='completed')


- **Resultado da primeira chamada: `arguments`**

Não há uma regra fixa, o modelo pode retornar em `arguments` a pergunta do usuário reformulada ou simplesmente copiar um query literal contido em `instructions` - nesse caso temos o `PUBMED_QUERY`.

In [27]:
call = response.output[0]
args = json.loads(call.arguments)

results = search_pubmed(**args)
result_json = json.dumps(results, indent=2)

result_json

'{\n  "header": {\n    "type": "esearch",\n    "version": "0.3"\n  },\n  "esearchresult": {\n    "count": "100",\n    "retmax": "20",\n    "retstart": "0",\n    "idlist": [\n      "41328123",\n      "41076314",\n      "40725551",\n      "40393895",\n      "40335417",\n      "40153056",\n      "40090550",\n      "39756382",\n      "39490003",\n      "39487073",\n      "39344460",\n      "39344453",\n      "39299812",\n      "39256068",\n      "39232864",\n      "38743036",\n      "38685147",\n      "38071499",\n      "37847424",\n      "37246472"\n    ],\n    "translationset": [],\n    "querytranslation": "(\\"Temporomandibular Joint\\"[MeSH Terms] OR \\"Temporomandibular Joint\\"[Title/Abstract]) AND (\\"Joint Prosthesis\\"[MeSH Terms] OR \\"prosthe*\\"[Title/Abstract] OR \\"replacement\\"[Title/Abstract] OR \\"alloplastic\\"[Title/Abstract]) AND (\\"infection*\\"[Title/Abstract] OR \\"antibiotic*\\"[Title/Abstract] OR \\"prophylaxis\\"[Title/Abstract]) AND 2000/01/01:2025/12/31[Date -

O objeto `function_call` inteiro (o `ResponseFunctionToolCall`) é o que entra em history, não só o texto ou os argumentos - é justamente esse objeto completo (com `.call_id`, `.name`, `.arguments`) que a API depois precisa ver de volta no histórico pra saber a quê o `function_call_output` que você manda em seguida está respondendo.

In [28]:
history += response.output

history.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool],
)

response.output_text

'The query returned **100 PubMed records total**.  \nThe first 20 PMIDs are:\n\n1. **41328123**\n2. **41076314**\n3. **40725551**\n4. **40393895**\n5. **40335417**\n6. **40153056**\n7. **40090550**\n8. **39756382**\n9. **39490003**\n10. **39487073**\n11. **39344460**\n12. **39344453**\n13. **39299812**\n14. **39256068**\n15. **39232864**\n16. **38743036**\n17. **38685147**\n18. **38071499**\n19. **37847424**\n20. **37246472**\n\nIf you want, I can also help format these into a deduplicated screening list or fetch titles/abstracts for them.'

## **Agentic loop**

In [29]:
def make_call(call):
    # JSON string -> python dict
    args = json.loads(call.arguments)

    if call.name == "search_pubmed":
        result = search_pubmed(**args)

    # Turn results into JSON
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [30]:
MAX_TOOL_CALLS = 2

instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=[search_pubmed_tool],
        )

        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result_summary': call_output['output'][:200],
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

## **Testando com três perguntas conhecidas**

A terceira pergunta não tem relação nenhuma com PubMed — serve pra confirmar que o agente também sabe **não** chamar a tool quando ela não é necessária.

In [31]:
known_questions = [
    "How many PMIDs does the query in effect for this pipeline return?",
    "What is the first PMID returned by our current search?",
    "What is the capital of France?",
]

for question in known_questions:
    print("=" * 60)
    print("QUESTION:", question)
    result = run_agent(question)
    print("STOP REASON:", result["stop_reason"])
    print("CALL LOG:", result["call_log"])
    print("ANSWER:", result["answer"])
    print()

QUESTION: How many PMIDs does the query in effect for this pipeline return?
iteration #0...
function_call: search_pubmed {"query":"( \"Temporomandibular Joint\"[Mesh] OR \"temporomandibular joint\"[Title/Abstract] ) AND ( \"Joint Prosthesis\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \"2000/01/01\"[Date - Publication] : \"2025/12/31\"[Date - Publication] )","limit":20}
iteration #1...
ASSISTANT:
The query returns **100 PMIDs** in total.
STOP REASON: end_turn
CALL LOG: [{'tool': 'search_pubmed', 'arguments': {'query': '( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/0